# D115 — Install and Use the MovieLens Wheel

This lab installs the three wheels created by D114 into a clean virtual environment:

1. all dependencies declared,
2. only `simplejson` declared,
3. no dependencies declared.

We reuse only this temporary environment:

```text
temp-venv
```

This is relative to the directory containing the notebook. Each demonstration deletes and recreates only that directory. No other course directory is deleted.

## 1. Prerequisite: run D114

D114 must create these local wheel files first.

In [ ]:
from pathlib import Path

wheel_root = Path(r"C:\tmp\movielens-wheel-lab")
wheel_name = "movielens_course-1.0.0-py3-none-any.whl"

full_wheel = wheel_root / "all-dependencies" / wheel_name
partial_wheel = wheel_root / "partial-dependencies" / wheel_name
no_deps_wheel = wheel_root / "no-dependencies" / wheel_name

for wheel in (full_wheel, partial_wheel, no_deps_wheel):
    print(wheel, "FOUND" if wheel.exists() else "MISSING")

If any wheel is missing, activate `dataeng`, run D114 completely, and return to this notebook.

## 2. Understand the commands

Run the following blocks in **Command Prompt**. Activating a virtual environment changes `python` and `pip` only in that Command Prompt window.

Important commands:

```bat
python -m venv temp-venv
temp-venv\Scripts\activate.bat
where python
python -m pip --version
deactivate
rmdir /S /Q temp-venv
```

After activation, `where python` should list `temp-venv\Scripts\python.exe` first.

## 3. Demonstration A — all dependencies declared

Start with a clean environment and install the full-dependency wheel:

```bat
:: Open Command Prompt in the directory containing this notebook
if exist temp-venv rmdir /S /Q temp-venv
python -m venv temp-venv
call temp-venv\Scripts\activate.bat

where python
python -m pip --version
python -m pip install "C:\tmp\movielens-wheel-lab\all-dependencies\movielens_course-1.0.0-py3-none-any.whl"
```

Pip reads both `Requires-Dist` entries and installs `simplejson`, `python-dateutil`, and the transitive `six` dependency automatically.

```bat
python -m pip show movielens-course
python -m pip show simplejson
python -m pip show python-dateutil
python -m pip check
python -c "from movielens.models import Movie, Rating; print(Movie); print(Rating)"
python -c "from movielens.writers.json_writer import JsonWriter; print(JsonWriter)"
```

Expected: both imports succeed and `pip check` reports no broken requirements.

You can also run the installed console command:

```bat
movielens-report
```

Then leave and remove the environment:

```bat
deactivate
rem Stay in the directory containing the notebook
rmdir /S /Q temp-venv
```

## 4. Demonstration B — one dependency declared

The partial wheel declares `simplejson` but omits `python-dateutil`:

```bat
:: Open Command Prompt in the directory containing this notebook
if exist temp-venv rmdir /S /Q temp-venv
python -m venv temp-venv
call temp-venv\Scripts\activate.bat

python -m pip install "C:\tmp\movielens-wheel-lab\partial-dependencies\movielens_course-1.0.0-py3-none-any.whl"
python -m pip show simplejson
python -m pip show python-dateutil
```

`simplejson` is installed automatically. `python-dateutil` is absent because the wheel metadata does not mention it.

Demonstrate the missing dependency:

```bat
python -c "from movielens.models import Rating"
```

Expected error:

```text
ModuleNotFoundError: No module named 'dateutil'
```

Notice that `pip check` cannot identify this undeclared dependency:

```bat
python -m pip check
```

Pip validates package metadata, not every import statement in the source.

Install the omitted dependency manually and retry:

```bat
python -m pip install python-dateutil
python -c "from movielens.models import Rating; print(Rating)"
python -c "from movielens.writers.json_writer import JsonWriter; print(JsonWriter)"
movielens-report
```

Clean up:

```bat
deactivate
rem Stay in the directory containing the notebook
rmdir /S /Q temp-venv
```

## 5. Demonstration C — no dependencies declared

The third wheel has an empty `install_requires` list:

```bat
:: Open Command Prompt in the directory containing this notebook
if exist temp-venv rmdir /S /Q temp-venv
python -m venv temp-venv
call temp-venv\Scripts\activate.bat

python -m pip install "C:\tmp\movielens-wheel-lab\no-dependencies\movielens_course-1.0.0-py3-none-any.whl"
python -m pip list
```

Neither external package is installed. First demonstrate the missing model dependency:

```bat
python -c "from movielens.models import Rating"
```

Expected: `ModuleNotFoundError: No module named 'dateutil'`.

Install it and verify the model:

```bat
python -m pip install python-dateutil
python -c "from movielens.models import Rating; print(Rating)"
```

Now demonstrate the missing writer dependency:

```bat
python -c "from movielens.writers.json_writer import JsonWriter"
```

Expected: `ModuleNotFoundError: No module named 'simplejson'`.

Install it and run the application:

```bat
python -m pip install simplejson
python -c "from movielens.writers.json_writer import JsonWriter; print(JsonWriter)"
python -m pip check
movielens-report
```

Because the no-dependency wheel provides no version constraint, manual `pip install simplejson` may select a newer major version than the full wheel's declared `simplejson>=3.19,<4` range. Missing dependency metadata also loses the package author's tested version boundaries.

Final cleanup:

```bat
deactivate
rem Stay in the directory containing the notebook
rmdir /S /Q temp-venv
if exist temp-venv (echo Cleanup failed) else (echo temp-venv removed)
```

## 6. Why the three installations differ

| Wheel variant | Declared metadata | Automatically installed | Manual work |
|---|---|---|---|
| All dependencies | `simplejson`, `python-dateutil` | Both plus transitive dependencies | None |
| Partial | `simplejson` | `simplejson` | Install `python-dateutil` |
| None | Empty | Nothing external | Install both packages |

The application source is identical in all three wheels. Only the `Requires-Dist` metadata changes.

## 7. Important observations

- A wheel does not normally contain its third-party dependencies.
- Pip installs only dependencies declared in wheel metadata.
- `pip check` cannot report an import that was never declared as a dependency.
- Test installations in a fresh environment so globally installed packages cannot hide missing metadata.
- Use `python -m pip` after activation to target the intended virtual environment.
- Delete only the known `temp-venv` directory during cleanup.